# Модуль 02 · Урок 1. Индексы, loc, iloc и срезы

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [1]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


pandas: 3.1.1 | корень проекта: c:\Users\304-18\Desktop\shapkina_tatiana_isip_30\pandas


## 1. Метки и позиции: две разные системы координат
В DataFrame строка имеет **метку индекса** (index label), а также **позицию** — номер её расположения в текущем порядке. Они могут совпадать, но это не правило. Например, после сортировки строки меняются местами, однако метки индекса обычно остаются прежними. При загрузке данных меткой может быть код заказа, дата или произвольное число. Поэтому привычка воспринимать `df.loc[2]` как «третью строку» приводит к ошибкам.

- `df.loc[метки_строк, метки_столбцов]` выбирает **по меткам**. Метки могут быть строками.
- `df.iloc[позиции_строк, позиции_столбцов]` выбирает **по целочисленным позициям**, начиная с нуля.
- `df.at[метка, столбец]` и `df.iat[позиция, позиция]` предназначены для одной ячейки.

In [2]:
import pandas as pd
warehouse = pd.DataFrame({'zone':['север','юг','запад'], 'stock':[12,7,24]}, index=['k9','k2','k7'])
print(warehouse.loc['k2', 'stock'])  # 7: метка k2
print(warehouse.iloc[1, 1])        # 7: позиция 1
print(warehouse.iloc[:2])          # первые две строки независимо от меток

7
7
     zone  stock
k9  север     12
k2     юг      7


## 2. Почему границы срезов различаются
В `iloc[start:stop]` правая граница **не включается**, как у обычного Python-списка. В `loc['k9':'k7']` обе границы обычно **включаются**: выбираем диапазон существующих меток в текущем порядке индекса. Срез `loc` зависит от того, где расположены метки, а при несортированных или повторяющихся индексах имеет дополнительные ограничения.

При выборе нескольких столбцов результат — DataFrame, если передать **список**: `df.loc[:, ['zone']]`. Если передать строковое имя — Series: `df.loc[:, 'zone']`. Это важно для последующей работы: к Series нельзя применить методы, ожидающие двумерный объект.

In [3]:
report = pd.DataFrame({'jan':[2,5,8,9], 'feb':[4,7,1,3]},index=['a','b','c','d'])
print(report.loc['b':'c', ['jan']])
print(report.iloc[1:3, 0:1])

   jan
b    5
c    8
   jan
b    5
c    8


## 3. Индекс не обязан быть уникальным
Если значения индекса повторяются, `loc['x']` может вернуть несколько строк. `set_index('column')` переносит столбец в индекс, а `reset_index()` возвращает обычную нумерацию. Существующий индекс может содержать полезную информацию; прежде чем его сбрасывать, решите, нужен ли он в итоговом результате.

In [4]:
visits = pd.DataFrame({'day':['пн','пн','вт'], 'users':[3,5,4]})
by_day = visits.set_index('day')
print(by_day.loc['пн'])  # две строки
print(by_day.reset_index())

     users
day       
пн       3
пн       5
  day  users
0  пн      3
1  пн      5
2  вт      4


## 4. Типичные ошибки
`KeyError` при `loc` означает, что метки нет. `IndexError` при `iloc` — позиция за пределами таблицы. `loc` с целым числом НЕ превращается автоматически в `iloc`. В pandas 3.x не изменяйте данные через цепочку `df['x'][mask] = ...`: изменение следует делать одним вызовом `.loc[mask, 'x'] = ...`. Позже разберём Copy-on-Write подробно.

**Проверьте себя:** что выберет `iloc[-1]`? Почему после сортировки `loc[0]` не обязательно является первой строкой? Чем отличаются `df[['price']]` и `df['price']`?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m2_01 · pick cells

Получите DataFrame с указанными метками строк `labels` и названиями колонок `columns` именно в заданном порядке.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `labels` — список меток строк; тип: `list`.
- `columns` — список имён столбцов; тип: `list`.


**Выполнение:** используйте готовые переменные `df`, `labels`, `columns`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [5]:
# Исходные данные задания m2_01: используются переменные df, labels, columns

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

labels = ['x', 'a']

columns = ['city', 'price']

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m2_01 = expected_for("m2_01", df=df, labels=labels, columns=columns)


df — демонстрационные данные:


,order_id,client_id,city,category,price,qty,discount,status,date,comment
z,demo_o1,1,Москва,чай,127.0,2,0,paid,2026-01-03,Быстрая доставка
b,demo_o2,2,Тверь,кофе,247.0,1,10,new,2026-01-04,без звонка
a,demo_o3,1,Москва,чай,97.0,4,5,paid,2026-01-04,Срочно
d,demo_o4,3,Казань,книги,317.0,1,0,paid,2026-02-01,подарок
x,demo_o5,2,Тверь,кофе,57.0,3,20,cancelled,2026-02-02,
q,demo_o6,4,Казань,чай,147.0,0,0,new,2026-02-05,NaN


In [6]:
# m2_01: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, labels, columns
# Сохраните конечный результат в переменную result.
result = df.loc[labels, columns]  # замените этой строкой/своим кодом
result

,city,price
x,Тверь,57.0
a,Москва,97.0


**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 2 строки, 2 столбца; колонки: `city`, `price`.

```text
     city  price
x   Тверь   57.0
a  Москва   97.0
```

In [7]:
# Проверка результата предыдущей ячейки
check_result("m2_01", result, _expected_m2_01)

✓ m2_01: верно! Результат совпадает с ожидаемым выводом.


True

### m2_02 · take positions

Получите последние `n` строк, но только первые три столбца. `n` всегда положительно.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `n` — число строк (`int`); в этом примере: `2`.


**Выполнение:** используйте готовые переменные `df`, `n`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [8]:
# Исходные данные задания m2_02: используются переменные df, n

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

n = 2

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m2_02 = expected_for("m2_02", df=df, n=n)


df — демонстрационные данные:


,order_id,client_id,city,category,price,qty,discount,status,date,comment
z,demo_o1,1,Москва,чай,127.0,2,0,paid,2026-01-03,Быстрая доставка
b,demo_o2,2,Тверь,кофе,247.0,1,10,new,2026-01-04,без звонка
a,demo_o3,1,Москва,чай,97.0,4,5,paid,2026-01-04,Срочно
d,demo_o4,3,Казань,книги,317.0,1,0,paid,2026-02-01,подарок
x,demo_o5,2,Тверь,кофе,57.0,3,20,cancelled,2026-02-02,
q,demo_o6,4,Казань,чай,147.0,0,0,new,2026-02-05,NaN


In [14]:
# m2_02: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, n
# Сохраните конечный результат в переменную result.
result = df.iloc[-n:, :3]  # замените этой строкой/своим кодом
result

,order_id,client_id,city
x,demo_o5,2,Тверь
q,demo_o6,4,Казань


**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 2 строки, 3 столбца; колонки: `order_id`, `client_id`, `city`.

```text
  order_id  client_id    city
x  demo_o5          2   Тверь
q  demo_o6          4  Казань
```

In [15]:
# Проверка результата предыдущей ячейки
check_result("m2_02", result, _expected_m2_02)

✓ m2_02: верно! Результат совпадает с ожидаемым выводом.


True

### m2_03 · single value

Извлеките скаляр из ячейки по метке строки и названию столбца.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.
- `label` — входной параметр (значение видно в примере); в этом примере: `'a'`.
- `column` — название столбца (`str`); в этом примере: `'price'`.


**Выполнение:** используйте готовые переменные `df`, `label`, `column`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [16]:
# Исходные данные задания m2_03: используются переменные df, label, column

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

label = 'a'

column = 'price'

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m2_03 = expected_for("m2_03", df=df, label=label, column=column)


df — демонстрационные данные:


,order_id,client_id,city,category,price,qty,discount,status,date,comment
z,demo_o1,1,Москва,чай,127.0,2,0,paid,2026-01-03,Быстрая доставка
b,demo_o2,2,Тверь,кофе,247.0,1,10,new,2026-01-04,без звонка
a,demo_o3,1,Москва,чай,97.0,4,5,paid,2026-01-04,Срочно
d,demo_o4,3,Казань,книги,317.0,1,0,paid,2026-02-01,подарок
x,demo_o5,2,Тверь,кофе,57.0,3,20,cancelled,2026-02-02,
q,demo_o6,4,Казань,чай,147.0,0,0,new,2026-02-05,NaN


In [23]:
# m2_03: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df, label, column
# Сохраните конечный результат в переменную result.
result = df.loc[label, column]  # замените этой строкой/своим кодом
result

np.float64(97.0)

**Ожидаемый вывод** (для данных из ячейки выше):

`float64`. Результат для показанного примера: `np.float64(97.0)`.

In [24]:
# Проверка результата предыдущей ячейки
check_result("m2_03", result, _expected_m2_03)

✓ m2_03: верно! Результат совпадает с ожидаемым выводом.


True

### m2_04 · index on id

Создайте новый DataFrame с индексом `order_id`, удалив столбец `order_id` из данных; исходник сохраните без изменений.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [25]:
# Исходные данные задания m2_04: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m2_04 = expected_for("m2_04", df=df)


df — демонстрационные данные:


,order_id,client_id,city,category,price,qty,discount,status,date,comment
z,demo_o1,1,Москва,чай,127.0,2,0,paid,2026-01-03,Быстрая доставка
b,demo_o2,2,Тверь,кофе,247.0,1,10,new,2026-01-04,без звонка
a,demo_o3,1,Москва,чай,97.0,4,5,paid,2026-01-04,Срочно
d,demo_o4,3,Казань,книги,317.0,1,0,paid,2026-02-01,подарок
x,demo_o5,2,Тверь,кофе,57.0,3,20,cancelled,2026-02-02,
q,demo_o6,4,Казань,чай,147.0,0,0,new,2026-02-05,NaN


In [26]:
# m2_04: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = df.set_index('order_id')  # замените этой строкой/своим кодом
result

,client_id,city,category,price,qty,discount,status,date,comment
order_id,,,,,,,,,
demo_o1,1,Москва,чай,127.0,2,0,paid,2026-01-03,Быстрая доставка
demo_o2,2,Тверь,кофе,247.0,1,10,new,2026-01-04,без звонка
demo_o3,1,Москва,чай,97.0,4,5,paid,2026-01-04,Срочно
demo_o4,3,Казань,книги,317.0,1,0,paid,2026-02-01,подарок
demo_o5,2,Тверь,кофе,57.0,3,20,cancelled,2026-02-02,
demo_o6,4,Казань,чай,147.0,0,0,new,2026-02-05,NaN


**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 9 столбцов; колонки: `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.

```text
          client_id    city category  price  qty  discount     status        date          comment
order_id                                                                                          
demo_o1           1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...
demo_o2           2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка
demo_o3           1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно
demo_o4           3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок
demo_o5           2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                 
demo_o6           4  Казань      чай  147.0    0         0        new  2026-02-05             None
```

In [27]:
# Проверка результата предыдущей ячейки
check_result("m2_04", result, _expected_m2_04)

✓ m2_04: верно! Результат совпадает с ожидаемым выводом.


True

### m2_05 · restore index

Сбросьте индекс и **удалите старые метки**: в результате должен быть RangeIndex, а колонки прежними.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `df` — `pd.DataFrame`, колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.


**Выполнение:** используйте готовые переменные `df`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [32]:
# Исходные данные задания m2_05: используются переменные df

df = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
}, index=['z', 'b', 'a', 'd', 'x', 'q'])

print("df — демонстрационные данные:")

display(df)


# Снимок правильного результата для автопроверки.
_expected_m2_05 = expected_for("m2_05", df=df)


df — демонстрационные данные:


,order_id,client_id,city,category,price,qty,discount,status,date,comment
z,demo_o1,1,Москва,чай,127.0,2,0,paid,2026-01-03,Быстрая доставка
b,demo_o2,2,Тверь,кофе,247.0,1,10,new,2026-01-04,без звонка
a,demo_o3,1,Москва,чай,97.0,4,5,paid,2026-01-04,Срочно
d,demo_o4,3,Казань,книги,317.0,1,0,paid,2026-02-01,подарок
x,demo_o5,2,Тверь,кофе,57.0,3,20,cancelled,2026-02-02,
q,demo_o6,4,Казань,чай,147.0,0,0,new,2026-02-05,NaN


In [33]:
# m2_05: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: df
# Сохраните конечный результат в переменную result.
result = df.reset_index(drop=True)  # замените этой строкой/своим кодом
result

,order_id,client_id,city,category,price,qty,discount,status,date,comment
0,demo_o1,1,Москва,чай,127.0,2,0,paid,2026-01-03,Быстрая доставка
1,demo_o2,2,Тверь,кофе,247.0,1,10,new,2026-01-04,без звонка
2,demo_o3,1,Москва,чай,97.0,4,5,paid,2026-01-04,Срочно
3,demo_o4,3,Казань,книги,317.0,1,0,paid,2026-02-01,подарок
4,demo_o5,2,Тверь,кофе,57.0,3,20,cancelled,2026-02-02,
5,demo_o6,4,Казань,чай,147.0,0,0,new,2026-02-05,NaN


**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 10 столбцов; колонки: `order_id`, `client_id`, `city`, `category`, `price`, `qty`, `discount`, `status`, `date`, `comment`.

```text
  order_id  client_id    city category  price  qty  discount     status        date          comment
0  demo_o1          1  Москва      чай  127.0    2         0       paid  2026-01-03    Быстрая до...
1  demo_o2          2   Тверь     кофе  247.0    1        10        new  2026-01-04       без звонка
2  demo_o3          1  Москва      чай   97.0    4         5       paid  2026-01-04           Срочно
3  demo_o4          3  Казань    книги  317.0    1         0       paid  2026-02-01          подарок
4  demo_o5          2   Тверь     кофе   57.0    3        20  cancelled  2026-02-02                 
5  demo_o6          4  Казань      чай  147.0    0         0        new  2026-02-05             None
```

In [34]:
# Проверка результата предыдущей ячейки
check_result("m2_05", result, _expected_m2_05)

✓ m2_05: верно! Результат совпадает с ожидаемым выводом.


True

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [36]:
show_progress()

Прогресс Pandas Academy:
  Модуль 01: 20/20
  Модуль 02: 5/20
  Модуль 03: 0/20
  Модуль 04: 0/20
  Модуль 05: 0/20
  Модуль 06: 0/20
  Модуль 07: 0/20
  Модуль 08: 0/15
ИТОГО: 25/155
